<a href="https://colab.research.google.com/github/Rageel-28/244107020136-Machine-Learning/blob/main/JS09/JS09_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS09 - Approximate Nearest Neighbors (ANN)
## Lab 3 - Exact NN vs. HNSW

**Tujuan:** membandingkan Exact NN (`scikit-learn`, *brute force*) dengan **HNSW** (*Hierarchical Navigable Small World*) dari library `hnswlib`.

HNSW menyusun data sebagai **graf berlapis**: lapisan atas jarang (untuk melompat jauh ke area *query*), lapisan bawah rapat (untuk penyempurnaan hasil). Pencarian bersifat *greedy*: bergerak ke tetangga yang lebih dekat dengan *query* sampai tidak ada yang lebih dekat.

## Langkah 1 - Instalasi Library

In [1]:
# Instal library bila belum ada (di Colab setara dengan !pip install ...)
import importlib, subprocess, sys

for modul, paket in [('hnswlib', 'hnswlib'), ('sklearn', 'scikit-learn')]:
    try:
        importlib.import_module(modul)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', paket])

print('Library siap dipakai')

Library siap dipakai


## Langkah 2 - Import Library

In [2]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

## Langkah 3 - Membuat Dataset Dummy

> Modul tidak mengatur *seed*, sehingga hasil berbeda tiap dijalankan. Di sini ditambahkan `np.random.seed(42)` agar hasil dapat direproduksi.

In [3]:
# Build Dummy Dataset
np.random.seed(42)
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # cari 5 tetangga terdekat

## Langkah 4 - Membuat Model NN (Exact NN)

In [4]:
# Build NN Model
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

# Compute time performance
start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "s")

=== Exact NN ===
Indices: [[137 170 750 316 991]]
Distances: [[0.01144331 0.0278216  0.02824018 0.03417073 0.04514859]]
Waktu: 0.042740821838378906 s


## Langkah 5 - Membuat Model HNSW

Parameter penting:
- `M` : jumlah koneksi maksimum tiap simpul pada graf (makin besar → lebih akurat, memori lebih besar).
- `ef_construction` : lebar pencarian saat membangun graf (makin besar → graf lebih baik, *build* lebih lambat).
- `ef` (`set_ef`) : lebar pencarian saat *query* (makin besar → lebih akurat, lebih lambat). **`ef` harus ≥ `k`.**

In [5]:
# Initiate Index
p = hnswlib.Index(space='l2', dim=dim)

# Define Max Elements
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Add Data
p.add_items(data)

# Set searching parameter
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances_hnsw = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances_hnsw)
print("Waktu:", end - start, "s")


=== HNSW ===
Indices: [[137 170 750 316 991]]
Distances: [[0.00013095 0.00077404 0.00079751 0.00116764 0.0020384 ]]
Waktu: 0.0002739429473876953 s


## Langkah 6 - Mengapa Jarak HNSW Tampak Berbeda?

Pada keluaran modul, tetangganya **sama** (`[102 238 579 373 486]`) tetapi angka jaraknya berbeda jauh (misal 0.0133 pada Exact NN vs 0.000177 pada HNSW). Ini **bukan** berarti HNSW salah: `hnswlib` dengan `space='l2'` mengembalikan **jarak Euclidean kuadrat** ($d^2$), sedangkan `scikit-learn` mengembalikan $d$. Buktinya, akar dari jarak HNSW harus sama dengan jarak Exact NN:

In [6]:
print('Jarak Exact NN (sklearn)      :', distances[0])
print('Jarak HNSW (kuadrat)          :', distances_hnsw[0])
print('Akar jarak HNSW               :', np.sqrt(distances_hnsw[0]))
print('Kuadrat jarak Exact NN        :', distances[0] ** 2)
print()
print('Indeks sama?                  :', list(indices[0]) == list(labels[0]))
print('Selisih maks (akar HNSW-Exact):', np.max(np.abs(np.sqrt(distances_hnsw[0]) - distances[0])))

Jarak Exact NN (sklearn)      : [0.01144331 0.0278216  0.02824018 0.03417073 0.04514859]
Jarak HNSW (kuadrat)          : [0.00013095 0.00077404 0.00079751 0.00116764 0.0020384 ]
Akar jarak HNSW               : [0.01144331 0.0278216  0.02824018 0.03417073 0.04514859]
Kuadrat jarak Exact NN        : [0.00013095 0.00077404 0.00079751 0.00116764 0.0020384 ]

Indeks sama?                  : True
Selisih maks (akar HNSW-Exact): 0.0


## Tambahan - Pengaruh `ef` pada Recall dan Waktu

Satu *query* tidak cukup untuk menilai kualitas. Berikut 200 *query* acak dan beberapa nilai `ef`; *recall@5* dihitung terhadap Exact NN. Nilai `ef` yang sama dengan `k` (5) adalah batas minimum.

In [7]:
import pandas as pd

rng = np.random.RandomState(0)
Q = rng.random_sample((200, dim)).astype(np.float32)

t0 = time.perf_counter()
_, I_gt = nn.kneighbors(Q)
t_exact = (time.perf_counter() - t0) * 1000 / len(Q)

baris = [{'Metode': 'Exact (sklearn brute)', 'ef': '-', 'Recall@5': 1.0, 'Query (ms/query)': t_exact}]
for ef in [5, 10, 20, 50, 100]:
    p.set_ef(ef)
    t0 = time.perf_counter()
    I_h, _ = p.knn_query(Q, k=k)
    t_avg = (time.perf_counter() - t0) * 1000 / len(Q)
    rec = np.mean([len(set(a) & set(b)) / k for a, b in zip(I_h, I_gt)])
    baris.append({'Metode': 'HNSW (M=16)', 'ef': ef, 'Recall@5': rec, 'Query (ms/query)': t_avg})

p.set_ef(50)  # kembalikan
pd.DataFrame(baris).round(5)

,Metode,ef,Recall@5,Query (ms/query)
0,Exact (sklearn brute),-,1.000,0.02432
1,HNSW (M=16),5,0.994,0.00283
2,HNSW (M=16),10,1.000,0.00662
3,HNSW (M=16),20,1.000,0.00543
4,HNSW (M=16),50,1.000,0.01030
5,HNSW (M=16),100,1.000,0.01887


### Analisis

- **Hasil sama.** Exact NN dan HNSW mengembalikan lima tetangga yang sama (`[137 170 750 316 991]`). Angka jarak HNSW tampak jauh lebih kecil hanya karena `hnswlib` (`space='l2'`) mengembalikan **jarak kuadrat**: akar jarak HNSW sama persis dengan jarak Exact NN (selisih maksimum 0.0).
- **Pengaruh `ef`** (200 *query*, recall@5):

| `ef` | Recall@5 |
|---|---|
| 5 (= `k`) | 0.994 |
| 10 | 1.000 |
| 20 | 1.000 |
| 50 | 1.000 |
| 100 | 1.000 |

  `ef` yang sama dengan `k` adalah batas minimum dan sudah hampir sempurna pada data 2D yang kecil ini. Di atas `ef=10` recall tidak naik lagi, sedangkan waktu *query* cenderung naik (±0.003 ms pada `ef=5` menjadi ±0.006 ms pada `ef=50`; angka waktu bergantung mesin dan berfluktuasi).
- **Kesimpulan:** pada data kecil berdimensi rendah, HNSW hampir selalu menemukan tetangga yang benar. Kerugian akurasinya baru terlihat pada data besar dan berdimensi tinggi (Lab 4 dan Lab 5). Nilai `ef` yang lebih besar dari yang dibutuhkan hanya membuang waktu.